# Одна программа: чтение, точная память, две точные машины, чтение без ответов

`research_m5/one.py` получает рассказы словами и ответы к ним. О чём рассказы, ей не говорят. В ней пять частей:

- **память**: вещи рассказа и для каждой пары вещей — предложения, где они названы вместе; хранится как есть;
- **чтение**: что эти предложения говорят о паре — один выбор из немногих. Его называет маленькая сеть по словам; у предложений, встреченных много раз, есть ещё своя строка (они помнятся дословно);
- **соединение**: две машины, обе точные после обучения. *Счётчик*: связь прибавляет −1, 0 или +1 к каждому из нескольких целых чисел, ответ — по знакам чисел в конце пути. *Таблица*: две связи подряд дают отношение с именем, путь соединяется разбором;
- **учение** только по ответам: выборы мягкие, градиент идёт сквозь машину; потом выборы делаются жёсткими и точно пересчитываются на учебных рассказах. Какую машину оставить, решают сами рассказы: обе учатся на рассказах с короткими путями и проверяются на рассказах с длинными;
- **чтение без ответов**: рассказы, о которых её спросят, счётчик читает заранее. Где предложение невиданного вида связывает две вещи, а между ними есть другой путь из понятых предложений, счётчик обязан сойтись по обоим путям — так незнакомая формулировка получает смысл без единого ответа.

Ячейки ниже запускают одну и ту же программу на чужих наборах: все двадцать задач bAbI (для рассказов, идущих во времени, в программе есть третья машина — «записи и вспоминание»), StepGame (плоскость, до десяти шагов; отдельно — словами авторов, которых не было в обучении) и CLUTRR (родство, текст писали люди). Первые ячейки видеокарты не требуют. Последние две — то, ради чего нужна видеокарта: предобученная сеть на месте читателя и трансформеры-соперники на тех же рассказах.

Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки.

In [ ]:
import os, subprocess
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
MINUTES = float(os.environ.get('MX_MINUTES', 0.6 if QUICK else 90))
WORK = ROOT + '/mx-one'
os.makedirs(WORK + '/clutrr', exist_ok=True)
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
!python -u {M5}/colab/mind.py --work {WORK} --fetch | tail -1
if not os.path.exists(f'{WORK}/StepGame'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ZhengxiangShi/StepGame {WORK}/StepGame && git -C {WORK}/StepGame sparse-checkout set Dataset/TrainVersion
RAW = 'https://raw.githubusercontent.com'
for name, where in (('gen23_train.csv', 'kliang5/CLUTRR_huggingface_dataset/main/gen_train23_test2to10/train.csv'),
                    ('gen23_test.csv', 'kliang5/CLUTRR_huggingface_dataset/main/gen_train23_test2to10/test.csv'),
                    ('gen234_test.csv', 'kliang5/CLUTRR_huggingface_dataset/main/gen_train234_test2to10/test.csv')) + \
                   tuple((f'{part}_100.csv', f'azreasoners/LLM-ASP/main/clutrr/data/clutrr_1.3/{part}_100.csv') for part in ('clean', 'supporting', 'irrelevant', 'disconnected')):
    if not os.path.exists(f'{WORK}/clutrr/{name}'):
        !wget -q -O {WORK}/clutrr/{name} {RAW}/{where}
small = '--most 1500 --sample 600 --begin 1 --readers 1 --steps 200 --few 200' if QUICK else ''
TESTS = 'gen23_test.csv,gen234_test.csv,clean_100.csv,supporting_100.csv,irrelevant_100.csv,disconnected_100.csv'
BABI = WORK + '/tasks_1-20_v1-2/en'

## Все двадцать задач bAbI одним запуском

Программа проходит папку задач и для каждой сама выбирает одну из трёх машин. Третья — «записи и вспоминание» (`recall.py`) — для рассказов, идущих во времени («где сейчас», «у кого», «сколько»); она учится на 30–40 рассказах. Для положения фигур (задача 17) программа оставляет счётчик, для поиска пути (задача 19) — таблицу; эти две учатся на тысяче вопросов. На процессоре вышло: шестнадцать задач из двадцати выше 95 %, средняя ошибка 7,8 %; не взяты задачи 4 (89 %), 10, 16 и 20. В статье о сетях с памятью (Sukhbaatar и др., 2015) при тысяче вопросов на задачу средняя ошибка 12,4 % по одним ответам (не взяты одиннадцать задач) и 6,7 % с разметкой нужных предложений; на задачах 17, 18 и 19 сеть по одним ответам давала 60, 91 и 17 %, здесь — 100, 100 и 100 %. Полчаса на процессоре.

In [ ]:
!cd {M5} && python one.py --babi {BABI} {'--tasks 1,17,19 ' + small if QUICK else ''} | grep -v "begun from\|  step\|mind [0-9]"

## StepGame: до десяти шагов на плоскости

Учат на 50 000 чистых рассказов с k от 1 до 5, проверяют на рассказах с помехами при k от 1 до 10. Лучшая сеть статьи набора: 86 % при k = 1 и 21 % при k = 10. Программа оставляет счётчик; на процессоре вышло 98 % при k = 1 и 95 % при k = 10, за пять минут.

In [ ]:
!cd {M5} && python one.py --stepgame {WORK}/StepGame/Dataset/TrainVersion {small} | grep -v "begun from"

## StepGame словами других авторов: чтение без ответов

Те же рассказы пересказаны шаблонами двенадцати авторов (`anew.py`): учебные — словами восьми авторов, проверочные — словами четырёх других, ни одно предложение проверки в обучении не встречалось. Сначала программа читает новые предложения по словам (на процессоре 78 % при k = 1 и 60 % при k = 10). Потом она читает проверочные рассказы без ответов: где новое предложение связывает две вещи, между которыми есть другой путь из понятых предложений, счётчик обязан сойтись по обоим путям, и это говорит, что предложение значит. После этого на процессоре вышло 100 % при каждом k (78 777 рассказов, три минуты).

In [ ]:
!cd {M5} && python anew.py --data {WORK}/StepGame/Dataset/TrainVersion --out {WORK}/anew --teach 1,2,3,4,5,6,7,8 --test 9,10,11,12 | tail -2
!cd {M5} && python one.py --stepgame {WORK}/anew {small} | grep -v "begun from"

## CLUTRR: родство, текст писали люди

Учат на рассказах из 2–3 звеньев, проверяют на 2–10; из предложений теста в обучении встречаются 5 %. Три проверки: тест первого разбиения (опубликовано: DSR-LM с RoBERTa 61 %, GPT-3 29 %), тест второго разбиения и CLUTRR 1.3 — версия с исправленными ошибками (GPT-3 с правилами, написанными вручную: 97, 84, 92 и 90 % по четырём видам рассказов). Программа оставляет таблицу; на процессоре вышло 56 и 58 % на двух тестах версии 1.0 и 90, 72, 78 и 75 % на CLUTRR 1.3.

In [ ]:
!cd {M5} && python one.py --clutrr {WORK}/clutrr --tests {TESTS} {small} | grep -v "  step"

## Видеокарта: предобученная сеть на месте читателя

Читатель — единственная часть с весами. Здесь на его место ставится RoBERTa-base, остальное без изменений. Вопрос: прибавляют ли веса читателю ответов. Короткая проба на процессоре (этап 42) прибавки не дала, и моё ожидание тогда не сбылось. Жду на CLUTRR 1.3 (без видов с шумом: строка clean_100) не меньше 90 %; если выйдет как у маленьких читателей, веса читателю на этой задаче не помогают.

In [ ]:
big = '--encoder hf-internal-testing/tiny-random-RobertaModel --readers 1 --steps 30 --most 1500 --few 200' if QUICK else '--encoder roberta-base --readers 1'
!cd {M5} && python one.py --clutrr {WORK}/clutrr --tests {TESTS} --only table {big} 2>&1 | grep -v "Warning\|Loading\|transformers\|  step"

## Видеокарта: трансформеры на тех же рассказах

`rival.py`: обычный трансформер (6 слоёв, ширина 256, около 5 млн весов) и зацикленный (один блок по кругу) учатся на тех же 50 000 рассказов StepGame и проверяются на тех же тестах. На процессоре за 75 минут они не выучили и учебные рассказы, так что вопрос, держат ли они длину, решается только здесь. Названо заранее: жду при k = 10 меньше 50 %; опровержение — 90 % и выше.

In [ ]:
tiny = '--width 64 --layers 2 --heads 4 --tried 100' if QUICK else ''
each = 0.3 if QUICK else MINUTES / 2
!cd {M5} && python rival.py --data {WORK}/StepGame/Dataset/TrainVersion --minutes {each} {tiny} | grep -v "^step"
!cd {M5} && python rival.py --data {WORK}/StepGame/Dataset/TrainVersion --minutes {each} --looped --turns 8 {tiny} | grep -v "^step"